# NDgpu — tri-S_N on GPU: device-resident DSA within-group iteration (Colab)

Measures the effect of running the **within-group (DSA-accelerated) source
iteration entirely on the GPU** for the level-scheduled tri-S_N solver
(`engine="levels"`). Previously each transport sweep round-tripped through the
host (source assembled on CPU, copied to device, swept, flux copied back), so a
small mesh spent most of its time in `cudaMemcpy`/synchronisation rather than
arithmetic. Now the sweep, the scattering source **and the DSA diffusion
correction** all operate on device arrays: the only host↔device traffic per
group per outer is the incoming fission/scatter source and the outgoing flux —
*not* two transfers per sweep. The DSA diffusion solve, which is a scipy sparse
LU on CPU, becomes a Jacobi-preconditioned CG on the device (`ndgpu.linalg.pcg`).

The claim under test: **the CPU→GPU crossover moves to a smaller mesh**, i.e.
the GPU wins at less refinement than before, because the fixed per-sweep
overhead that used to dominate small problems is gone.

(Cross sections are illustrative placeholders, not predictive.)

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

## Method

* **Engine**: `engine="levels"`, SCB differencing, drums inserted, on the 2D
  HP-MR core. The mesh-refinement level `refine` is the problem-size axis.
* **Device-resident inner loop**: the new path — `TriSNTransportSolver`
  automatically runs `_solve_group_dev` on the levels engine, keeping `phi`,
  the scatter source `Σs·φ`, the sweep output and the DSA correction all in
  device memory across every inner iteration. On NumPy this is the identical
  arithmetic on the CPU, so CPU and GPU still run the **same iteration
  sequence** and the speed-up ratio is tolerance-independent.
* **CUDA graph**: each level loop is captured once per (group, iface) and
  replayed as a single launch. The `cudagraph` column reports whether capture
  engaged (`on`) or permanently fell back to the plain loop (`fallback`).
* **Warm-up**: one small untimed GPU solve first (compiles kernels, allocates
  the pool, captures graphs).
* **Timing**: `solve_seconds` is honest wall time — the outer's `k`/flux
  reductions still synchronise once per outer.

In [ ]:
import time
import numpy as np

from ndgpu.benchmarks import build_hpmr2d, hpmr_transport_mask
from ndgpu.tri_sn import TriSNTransportSolver

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))
REFINES = [2, 3] if QUICK else [2, 4, 6, 8, 10]
TOL = dict(tol_k=5e-7, tol_source=5e-6, max_outer=200)
QUAD = dict(n_polar=2, n_azi=8)


def problem(refine):
    p = build_hpmr2d(refine=refine, drum_angle_deg=0.0, absorber="polar")
    mix = dict(mix_material=p.mix_material, mix_weight=p.mix_weight)
    return p, mix


def run(refine, device):
    p, mix = problem(refine)
    t = time.perf_counter()
    s = TriSNTransportSolver(p.grid, p.materials, p.material_map,
                             active=p.active, bc="vacuum", scheme="scb",
                             engine="levels", device=device, **QUAD, **mix)
    setup = time.perf_counter() - t
    r = s.solve(**TOL)
    assert r.converged, f"refine={refine} device={device} did not converge"
    return dict(refine=refine, cells=int(p.active.sum()), k=r.k_eff,
                outers=r.outer_iterations, sweeps=r.n_sweeps,
                setup=setup, solve=r.solve_seconds, graphs=s.graphs_active,
                t_groups=s.t_groups, t_cmfd=s.t_cmfd, t_power=s.t_power)


if HAVE_GPU:                                # warm-up: compile kernels + capture graphs
    run(2, "gpu")

In [ ]:
rows = []
for refine in REFINES:
    cpu = run(refine, "cpu")
    row = dict(cpu)                                  # cpu component times: bare keys
    if HAVE_GPU:
        gpu = run(refine, "gpu")
        row.update(k_gpu=gpu["k"], t_gpu=gpu["solve"],
                   speedup=cpu["solve"] / gpu["solve"], graphs=gpu["graphs"],
                   g_solve=gpu["solve"], g_groups=gpu["t_groups"],
                   g_cmfd=gpu["t_cmfd"], g_power=gpu["t_power"],
                   g_sweeps=gpu["sweeps"])
    rows.append(row)

hdr = (f"{'refine':>6} {'cells':>7} {'outers':>6} {'sweeps':>7} "
       f"{'k':>10} {'t_cpu[s]':>9}")
if HAVE_GPU:
    hdr += (f" {'t_gpu[s]':>9} {'dpcm':>6} {'speedup':>8} {'cudagraph':>9}"
            f" {'gpu_swp':>7}")
print(hdr)
for r in rows:
    line = (f"{r['refine']:>6d} {r['cells']:>7d} {r['outers']:>6d} "
            f"{r['sweeps']:>7d} {r['k']:>10.6f} {r['solve']:>9.2f}")
    if HAVE_GPU:
        cg = {True: 'on', False: 'fallback'}.get(r.get('graphs'), '?')
        line += (f" {r['t_gpu']:>9.2f} {(r['k_gpu']-r['k'])*1e5:>6.2f} "
                 f"{r['speedup']:>8.2f} {cg:>9} {r.get('g_sweeps', 0):>7d}")
    print(line)

def breakdown(tag, ks, kg, kc, kp):
    print(f"\n{tag} time breakdown (transport groups vs CMFD; power* = host-only):")
    print(f"{'refine':>6} {'solve[s]':>9} {'groups[s]':>10} {'cmfd[s]':>9} "
          f"{'power*[s]':>10} {'group_share':>11}")
    for r in rows:
        if r.get(kg) is None:
            continue
        share = r[kg] / r[ks] if r[ks] else 0.0
        print(f"{r['refine']:>6d} {r[ks]:>9.2f} {r[kg]:>10.2f} {r[kc]:>9.2f} "
              f"{r[kp]:>10.2f} {share:>10.0%}")

breakdown('CPU', 'solve', 't_groups', 't_cmfd', 't_power')
if HAVE_GPU:
    breakdown('GPU', 'g_solve', 'g_groups', 'g_cmfd', 'g_power')

if HAVE_GPU:
    wins = [r for r in rows if r['speedup'] > 1.0]
    if wins:
        c = min(r['cells'] for r in wins)
        print(f"\nGPU first wins at {c} active cells "
              f"(refine={min(r['refine'] for r in wins if r['cells']==c)}).")
    else:
        print("\nGPU did not win at any tested refine -- try larger REFINES.")
    if HAVE_GPU and any(r.get('graphs') is False for r in rows):
        # capture fell back to the plain loop -- rebuild one solver and show why
        from ndgpu.tri_sn import TriSNTransportSolver
        p, mix = problem(REFINES[0])
        s = TriSNTransportSolver(p.grid, p.materials, p.material_map,
                                 active=p.active, bc="vacuum", scheme="scb",
                                 engine="levels", device="gpu", **QUAD, **mix)
        s.solve(**TOL)
        print("cudagraph fallback reason:", s._graph_error)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
cells_x = [r['cells'] for r in rows]
ax[0].plot(cells_x, [r['solve'] for r in rows], 'o-', label='CPU')
if HAVE_GPU:
    ax[0].plot(cells_x, [r['t_gpu'] for r in rows], 's-', label='GPU')
ax[0].set(xlabel='active cells', ylabel='solve wall time [s]',
          xscale='log', yscale='log', title='cost vs problem size')
ax[0].legend(); ax[0].grid(True, which='both', alpha=0.3)

if HAVE_GPU:
    ax[1].axhline(1.0, color='k', lw=0.8, ls='--')
    ax[1].plot(cells_x, [r['speedup'] for r in rows], 'D-', color='C2')
    ax[1].set(xlabel='active cells', ylabel='GPU speed-up (x)',
              xscale='log', title='CPU/GPU crossover')
    ax[1].grid(True, which='both', alpha=0.3)
else:
    ax[1].set_title('(run on a GPU runtime for the speed-up panel)')
plt.show()

## Reading the results

* **`speedup > 1` at a smaller `cells`** than the pre-port crossover is the
  win: eliminating the per-sweep host↔device transfers removes the fixed
  overhead that used to swamp small meshes, so the GPU pays off with less
  refinement.
* **`cudagraph = on`** confirms the level loop is replayed as a single captured
  launch. `fallback` means capture failed (the run is still correct, just
  without the launch-collapse); `?` means no GPU row.
* **`dpcm`** (GPU k − CPU k, in pcm) should be a handful of pcm or less: CPU and
  GPU run the identical iteration sequence, so any gap is only device
  floating-point associativity, not a modelling difference.
* **`outers`/`sweeps`** are device-independent (same schedule on both), so they
  are the honest denominator for the wall-time comparison.

The remaining host-side work per outer is the **CMFD** eigen-update (a diffusion
power iteration with no transport sweeps); porting that is a separate step and
only matters once its host compute — not the transport sweeps — dominates.

## Experiment ii — tuning the device DSA solve

On the GPU the within-group **DSA correction** is a Jacobi-preconditioned CG on
the device (the CPU uses an exact cached sparse LU). Run to a tight tolerance it
does hundreds of synced iterations and cancels the sweep speed-up — so it is
run as an *inexact* accelerator: loose `dsa_rtol`, capped `dsa_maxiter`, and the
convergence test spaced out (each test is a GPU sync). A weaker solve makes each
DSA cheaper but the source iteration take more (cheap, graphed) sweeps — this
sweeps `dsa_maxiter` at one mesh to find the wall-time optimum. The
source-iteration watchdog keeps a too-weak solve from ever hurting correctness.

In [ ]:
if HAVE_GPU:
    from ndgpu.tri_sn import TriSNTransportSolver
    TUNE_REFINE = 3 if QUICK else 8
    p, mix = problem(TUNE_REFINE)
    print(f"DSA tuning at refine={TUNE_REFINE} ({int(p.active.sum())} cells), GPU:")
    print(f"{'dsa_maxiter':>11} {'dsa_rtol':>9} {'solve[s]':>9} {'groups[s]':>10} "
          f"{'sweeps':>7} {'k':>10}")
    for maxit, rtol in [(50, 1e-3), (100, 1e-4), (200, 1e-6), (2000, 1e-9)]:
        s = TriSNTransportSolver(p.grid, p.materials, p.material_map,
                                 active=p.active, bc="vacuum", scheme="scb",
                                 engine="levels", device="gpu", **QUAD, **mix,
                                 dsa_maxiter=maxit, dsa_rtol=rtol)
        r = s.solve(**TOL)
        print(f"{maxit:>11d} {rtol:>9.0e} {r.solve_seconds:>9.2f} "
              f"{s.t_groups:>10.2f} {r.n_sweeps:>7d} {r.k_eff:>10.6f}")
    print("\n(last row ~ the old tight-solve behaviour; lower solve[s] is better)")
else:
    print("run on a GPU runtime for the DSA tuning sweep")